# 03 — Class Comparison

Compare damage output across Zuluhotel Omega classes.

All defined classes are pre-configured below with their in-class skills
(from `classes.inc`). Edit the **Configuration** cell to:
- Select which classes to include in the comparison
- Set the in-class skill level (applied to all skills in each class)
- Set the base stats (STR / DEX / INT) per class
- Choose the shared weapon and target

In [ ]:
import logging
from pathlib import Path

from omega.model.constants import (
    SKILLID_ALCHEMY, SKILLID_ANATOMY, SKILLID_ANIMALLORE, SKILLID_ARCHERY,
    SKILLID_ARMSLORE, SKILLID_BEGGING, SKILLID_BLACKSMITHY, SKILLID_BOWCRAFT,
    SKILLID_CAMPING, SKILLID_CARPENTRY, SKILLID_CARTOGRAPHY, SKILLID_COOKING,
    SKILLID_DETECTINGHIDDEN, SKILLID_ENTICEMENT, SKILLID_EVALINT,
    SKILLID_FENCING, SKILLID_FISHING, SKILLID_FORENSICS, SKILLID_HEALING,
    SKILLID_HERDING, SKILLID_HIDING, SKILLID_INSCRIPTION, SKILLID_ITEMID,
    SKILLID_LOCKPICKING, SKILLID_LUMBERJACKING, SKILLID_MACEFIGHTING,
    SKILLID_MAGERY, SKILLID_MAGICRESISTANCE, SKILLID_MEDITATION,
    SKILLID_MINING, SKILLID_MUSICIANSHIP, SKILLID_PARRY, SKILLID_PEACEMAKING,
    SKILLID_POISONING, SKILLID_PROVOCATION, SKILLID_REMOVETRAP,
    SKILLID_SNOOPING, SKILLID_SPIRITSPEAK, SKILLID_STEALTH, SKILLID_STEALING,
    SKILLID_SWORDSMANSHIP, SKILLID_TACTICS, SKILLID_TAILORING,
    SKILLID_TAMING, SKILLID_TASTEID, SKILLID_TINKERING, SKILLID_TRACKING,
    SKILLID_VETERINARY, SKILLID_WRESTLING,
)
from omega.shard import ShardData
from omega.simulation import (
    ArmorSpec, CombatantSpec, Scenario, WeaponSpec, run_scenario,
)
from omega.reporting.tables import comparison_table, format_table_html
from omega.reporting.plots import comparison_overlay
from omega.logging import setup_logging

# Suppress noisy stub warnings — only show errors in notebook output
setup_logging(level=logging.ERROR)

SHARD_ROOT = Path("submodules/zuluhotel_omega_2.5")
if not SHARD_ROOT.exists():
    SHARD_ROOT = Path("../submodules/zuluhotel_omega_2.5")
shard = ShardData.from_path(SHARD_ROOT)

## Class Definitions

Each class has its in-class skill set (from `classes.inc`). These are
used to build the skills dict — all in-class skills get the configured
skill level, all others stay at 0.

In [ ]:
# In-class skill sets from classes.inc GetClasseSkills()
CLASS_SKILLS = {
    "Warrior": [
        SKILLID_ANATOMY, SKILLID_FENCING, SKILLID_HEALING,
        SKILLID_MACEFIGHTING, SKILLID_PARRY, SKILLID_SWORDSMANSHIP,
        SKILLID_TACTICS, SKILLID_WRESTLING,
    ],
    "Mage": [
        SKILLID_ALCHEMY, SKILLID_EVALINT, SKILLID_INSCRIPTION,
        SKILLID_ITEMID, SKILLID_MAGERY, SKILLID_MEDITATION,
        SKILLID_MAGICRESISTANCE, SKILLID_SPIRITSPEAK,
    ],
    "Thief": [
        SKILLID_DETECTINGHIDDEN, SKILLID_HIDING, SKILLID_LOCKPICKING,
        SKILLID_POISONING, SKILLID_REMOVETRAP, SKILLID_SNOOPING,
        SKILLID_STEALING, SKILLID_STEALTH,
    ],
    "Bard": [
        SKILLID_BEGGING, SKILLID_CARTOGRAPHY, SKILLID_ENTICEMENT,
        SKILLID_HERDING, SKILLID_MUSICIANSHIP, SKILLID_PEACEMAKING,
        SKILLID_PROVOCATION, SKILLID_TASTEID,
    ],
    "Ranger": [
        SKILLID_ANIMALLORE, SKILLID_TAMING, SKILLID_ARCHERY,
        SKILLID_CAMPING, SKILLID_COOKING, SKILLID_FISHING,
        SKILLID_TRACKING, SKILLID_VETERINARY,
    ],
    "Paladin": [
        SKILLID_MACEFIGHTING, SKILLID_PARRY, SKILLID_SWORDSMANSHIP,
        SKILLID_TACTICS, SKILLID_MAGERY, SKILLID_EVALINT,
        SKILLID_MEDITATION, SKILLID_MAGICRESISTANCE,
    ],
    "Mystic Archer": [
        SKILLID_ARCHERY, SKILLID_COOKING, SKILLID_EVALINT,
        SKILLID_FISHING, SKILLID_TRACKING, SKILLID_MAGERY,
        SKILLID_MAGICRESISTANCE, SKILLID_MEDITATION,
    ],
    "Bladesinger": [
        SKILLID_ANATOMY, SKILLID_ENTICEMENT, SKILLID_FENCING,
        SKILLID_MUSICIANSHIP, SKILLID_PEACEMAKING, SKILLID_PROVOCATION,
        SKILLID_SWORDSMANSHIP, SKILLID_TACTICS,
    ],
    "Crafter": [
        SKILLID_ARMSLORE, SKILLID_BLACKSMITHY, SKILLID_BOWCRAFT,
        SKILLID_CARPENTRY, SKILLID_LUMBERJACKING, SKILLID_MINING,
        SKILLID_TAILORING, SKILLID_TINKERING,
    ],
    "Powerplayer": [
        # All skills from Warrior + Thief + Ranger + Mage + Crafter + Bard + Forensics
        SKILLID_ANATOMY, SKILLID_FENCING, SKILLID_HEALING,
        SKILLID_MACEFIGHTING, SKILLID_PARRY, SKILLID_SWORDSMANSHIP,
        SKILLID_TACTICS, SKILLID_WRESTLING,
        SKILLID_DETECTINGHIDDEN, SKILLID_HIDING, SKILLID_LOCKPICKING,
        SKILLID_POISONING, SKILLID_REMOVETRAP, SKILLID_SNOOPING,
        SKILLID_STEALING, SKILLID_STEALTH,
        SKILLID_ANIMALLORE, SKILLID_TAMING, SKILLID_ARCHERY,
        SKILLID_CAMPING, SKILLID_COOKING, SKILLID_FISHING,
        SKILLID_TRACKING, SKILLID_VETERINARY,
        SKILLID_ALCHEMY, SKILLID_EVALINT, SKILLID_INSCRIPTION,
        SKILLID_ITEMID, SKILLID_MAGERY, SKILLID_MEDITATION,
        SKILLID_MAGICRESISTANCE, SKILLID_SPIRITSPEAK,
        SKILLID_ARMSLORE, SKILLID_BLACKSMITHY, SKILLID_BOWCRAFT,
        SKILLID_CARPENTRY, SKILLID_LUMBERJACKING, SKILLID_MINING,
        SKILLID_TAILORING, SKILLID_TINKERING,
        SKILLID_BEGGING, SKILLID_CARTOGRAPHY, SKILLID_ENTICEMENT,
        SKILLID_HERDING, SKILLID_MUSICIANSHIP, SKILLID_PEACEMAKING,
        SKILLID_PROVOCATION, SKILLID_TASTEID,
        SKILLID_FORENSICS,
    ],
}

# Class ID used by the shard (GetObjProperty key → level)
CLASS_IDS = {
    "Warrior": "IsWarrior",
    "Mage": "IsMage",
    "Thief": "IsThief",
    "Bard": "IsBard",
    "Ranger": "IsRanger",
    "Paladin": "IsPaladin",
    "Mystic Archer": "IsMysticArcher",
    "Bladesinger": "IsBladesinger",
    "Crafter": "IsCrafter",
    "Powerplayer": "IsPowerplayer",
}

print(f"Defined {len(CLASS_SKILLS)} classes: {', '.join(CLASS_SKILLS)}")

## Configuration

Edit this cell to select classes, skill levels, stats, weapon, and target.

In [ ]:
# --- Classes to include (comment out any you want to exclude) ---
CLASSES_TO_COMPARE = [
    "Warrior",
    "Mage",
    # "Thief",
    # "Bard",
    "Ranger",
    "Paladin",
    # "Mystic Archer",
    "Bladesinger",
    # "Crafter",
    "Powerplayer",
]

# --- In-class skill level (applied to ALL skills in the class) ---
# Override per class, or set a default for all.
DEFAULT_SKILL_LEVEL = 100
CLASS_SKILL_LEVEL = {
    # "Warrior": 120,     # uncomment to override a specific class
    # "Mage": 110,
}

# --- Base stats per class (STR / DEX / INT) ---
# Override per class, or use defaults.
DEFAULT_STATS = {"str": 100, "dex": 100, "int": 25}
CLASS_STATS = {
    "Warrior":       {"str": 120, "dex":  80, "int": 25},
    "Mage":          {"str":  50, "dex":  50, "int": 130},
    "Thief":         {"str":  80, "dex": 120, "int": 25},
    "Bard":          {"str":  60, "dex": 100, "int": 65},
    "Ranger":        {"str":  90, "dex": 110, "int": 25},
    "Paladin":       {"str": 110, "dex":  80, "int": 35},
    "Mystic Archer": {"str":  60, "dex": 100, "int": 65},
    "Bladesinger":   {"str": 100, "dex": 100, "int": 25},
    "Crafter":       {"str": 100, "dex": 100, "int": 25},
    "Powerplayer":   {"str": 100, "dex": 100, "int": 25},
}

# --- Class level (1-5, used for CLASSE_BONUS calculation) ---
CLASS_LEVEL = 5

# --- Shared weapon and target ---
WEAPON = WeaponSpec(name="Broadsword", damage="3d6+2")
DEFENDER = CombatantSpec(
    name="Target", is_npc=True,
    str_=50, dex_=50, int_=50, hp=500,
    skills={SKILLID_WRESTLING: 80},  # skilled NPC — affects hit chance
    armor=ArmorSpec(ar=30),
)

# --- Simulation settings ---
ITERATIONS = 200
BASE_SEED = 42

print(f"Will compare: {', '.join(CLASSES_TO_COMPARE)}")
print(f"Default skill level: {DEFAULT_SKILL_LEVEL}, class level: {CLASS_LEVEL}")
print(f"Weapon: {WEAPON.name} ({WEAPON.damage}), Target AR: {DEFENDER.armor.ar}")
print(f"Defender Wrestling: {DEFENDER.skills.get(SKILLID_WRESTLING, 0)}")

## Run Simulation

In [ ]:
RUN_KW = dict(shard=shard)

results = {}
for cls_name in CLASSES_TO_COMPARE:
    skill_level = CLASS_SKILL_LEVEL.get(cls_name, DEFAULT_SKILL_LEVEL)
    stats = CLASS_STATS.get(cls_name, DEFAULT_STATS)
    skills = {sid: skill_level for sid in CLASS_SKILLS[cls_name]}

    spec = CombatantSpec(
        name=cls_name,
        skills=skills,
        str_=stats["str"],
        dex_=stats["dex"],
        int_=stats["int"],
        class_levels={CLASS_IDS[cls_name]: CLASS_LEVEL},
        weapon=WEAPON,
    )

    results[cls_name] = run_scenario(
        Scenario(attacker=spec, defender=DEFENDER, iterations=ITERATIONS, base_seed=BASE_SEED),
        **RUN_KW,
    )
    ds = results[cls_name].damage_stats
    r = results[cls_name].ratios
    print(f"  {cls_name:20s}  mean={ds.mean:6.2f}  on_hit={results[cls_name].damage_stats_on_hit.mean:6.2f}  hit_rate={r.hit_rate:.1%}")

print(f"\nSimulated {len(results)} classes x {ITERATIONS} iterations each.")

In [ ]:
# Overlaid damage distributions
comparison_overlay(
    results,
    title="Class Comparison — Damage Distribution",
)

In [ ]:
# Side-by-side comparison table
from IPython.display import HTML

rows = comparison_table(
    results,
    stats=["mean", "mean_on_hit", "median", "min", "max", "p5", "p95",
           "hit_rate", "absorbed_mean"],
)
HTML(format_table_html(rows))

## Reactive Armor Comparison (V1.5)

Re-run the same class comparison but with the defender using Reactive Armor.
Reactive Armor reflects `basedamage * power / 100` back to the attacker on
every hit (it's a one-shot buff, consumed after activation). The `power`
parameter controls reflection strength — here we use `power = 86` (~86% of
base damage reflected, reduced to 1/8 against player attackers).

The comparison table below shows **with vs without** RA so you can see the
reactive damage portion per class.

In [ ]:
import dataclasses
import math

# Power = floor((1 - pi) * 100) ≈ 86 — a non-trivial reflection strength
RA_POWER = math.floor((1 - math.pi / 4) * 100)  # ~21% of basedamage reflected

# Run both with and without RA for direct comparison
DEFENDER_RA = dataclasses.replace(DEFENDER, properties={"ReactiveArmor": RA_POWER})

ra_results = {}
for cls_name in CLASSES_TO_COMPARE:
    skill_level = CLASS_SKILL_LEVEL.get(cls_name, DEFAULT_SKILL_LEVEL)
    stats = CLASS_STATS.get(cls_name, DEFAULT_STATS)
    skills = {sid: skill_level for sid in CLASS_SKILLS[cls_name]}

    spec = CombatantSpec(
        name=cls_name,
        skills=skills,
        str_=stats["str"],
        dex_=stats["dex"],
        int_=stats["int"],
        class_levels={CLASS_IDS[cls_name]: CLASS_LEVEL},
        weapon=WEAPON,
    )

    # With RA
    ra_results[f"{cls_name} (RA)"] = run_scenario(
        Scenario(attacker=spec, defender=DEFENDER_RA, iterations=ITERATIONS, base_seed=BASE_SEED),
        **RUN_KW,
    )
    # Without RA (reuse from earlier)
    ra_results[f"{cls_name} (plain)"] = results[cls_name]

    ds_ra = ra_results[f"{cls_name} (RA)"].damage_stats
    ds_plain = results[cls_name].damage_stats
    r = ra_results[f"{cls_name} (RA)"].ratios
    print(f"  {cls_name:20s}  plain={ds_plain.mean:6.2f}  RA={ds_ra.mean:6.2f}  "
          f"reactive_rate={r.reactive_rate:.0%}  RA_power={RA_POWER}")

In [ ]:
# Comparison table: with vs without reactive armor side by side
rows = comparison_table(
    ra_results,
    stats=["mean", "mean_on_hit", "median", "p5", "p95", "hit_rate",
           "reactive_rate", "reactive_rate_on_hit", "absorbed_mean"],
)
HTML(format_table_html(rows))